# Parquet Basics — 06: Schema Evolution

## What you will learn
Data schemas change over time — new columns are added, old ones removed.
Parquet handles many schema changes gracefully through **schema evolution**.

In this notebook:
1. Adding columns — backward compatible, safe
2. Dropping columns — forward compatible
3. Renaming columns — requires care
4. Type promotion — safe vs unsafe changes
5. `mergeSchema` — handling multiple file versions
6. Schema registry pattern — tracking schema history


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 21:16:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/parquet_basics


26/04/10 21:17:04 WARN TaskSetManager: Stage 0 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:17:06 WARN TaskSetManager: Stage 1 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


Dataset: 200,000 rows | 12 columns
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- order_date: date (nullable = true)



## Step 1 — Adding Columns (Backward Compatible)

Adding a column with a default value is the safest schema change.
Old files simply return null for the new column.


In [2]:
BASE_PATH = f"{DATA_DIR}/schema_evolution"

# V1: original schema
v1_df = df.select("order_id","customer_id","product","category","revenue","order_date")
v1_path = f"{BASE_PATH}/v1"
v1_df.write.mode("overwrite").parquet(v1_path)
print(f"V1 schema: {v1_df.columns}")
print(f"V1 rows  : {v1_df.count():,}")

# V2: added 'discount' and 'country' columns
v2_df = df.select("order_id","customer_id","product","category",
                   "revenue","order_date","discount","country")
v2_path = f"{BASE_PATH}/v2"
v2_df.write.mode("overwrite").parquet(v2_path)
print(f"\nV2 schema: {v2_df.columns}")
print(f"V2 rows  : {v2_df.count():,}")

# Read V1 files with V2 schema (new cols = null)
v2_schema = spark.read.parquet(v2_path).schema
df_v1_as_v2 = spark.read.schema(v2_schema).parquet(v1_path)
print(f"\nV1 data read with V2 schema (new columns = null):")
df_v1_as_v2.select("order_id","revenue","discount","country").show(5)
print("discount and country are null for V1 rows — safe!")

26/04/10 21:17:07 WARN TaskSetManager: Stage 4 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

V1 schema: ['order_id', 'customer_id', 'product', 'category', 'revenue', 'order_date']


26/04/10 21:17:09 WARN TaskSetManager: Stage 5 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


V1 rows  : 200,000


26/04/10 21:17:09 WARN TaskSetManager: Stage 8 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.



V2 schema: ['order_id', 'customer_id', 'product', 'category', 'revenue', 'order_date', 'discount', 'country']


26/04/10 21:17:10 WARN TaskSetManager: Stage 9 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


V2 rows  : 200,000

V1 data read with V2 schema (new columns = null):
+--------+--------+--------+-------+
|order_id| revenue|discount|country|
+--------+--------+--------+-------+
|       1|  195.44|    NULL|   NULL|
|       2|  174.07|    NULL|   NULL|
|       3|14534.56|    NULL|   NULL|
|       4| 1948.71|    NULL|   NULL|
|       5| 1862.01|    NULL|   NULL|
+--------+--------+--------+-------+
only showing top 5 rows
discount and country are null for V1 rows — safe!


In [3]:
# Read both V1 and V2 together using mergeSchema
combined = spark.read.option("mergeSchema", True).parquet(v1_path, v2_path)
print(f"Combined schema (V1 + V2 union): {combined.columns}")
print(f"Total rows: {combined.count():,}")
print()
combined.select("order_id","revenue","discount","country").show(8)
print("V1 rows show null for discount/country — V2 rows show real values")

Combined schema (V1 + V2 union): ['order_id', 'customer_id', 'product', 'category', 'revenue', 'order_date', 'discount', 'country']
Total rows: 400,000

+--------+--------+--------+-------+
|order_id| revenue|discount|country|
+--------+--------+--------+-------+
|       1|  195.44|    0.11|     UK|
|       2|  174.07|   0.093|     AU|
|       3|14534.56|   0.003|     JP|
|       4| 1948.71|   0.152|     US|
|       5| 1862.01|   0.117|     FR|
|       6| 7501.16|   0.394|     BR|
|       7| 4532.55|   0.107|     DE|
|       8| 4962.43|   0.108|     CA|
+--------+--------+--------+-------+
only showing top 8 rows
V1 rows show null for discount/country — V2 rows show real values


## Step 2 — Removing Columns

When you stop writing a column, old files still have it.
New files won't have it. Reading with mergeSchema handles this.


In [4]:
# V3: removed 'discount', added 'loyalty_points'
v3_df = df.select("order_id","customer_id","product","category",
                   "revenue","order_date","country") \
           .withColumn("loyalty_points", (F.col("revenue") * 10).cast("int"))

v3_path = f"{BASE_PATH}/v3"
v3_df.write.mode("overwrite").parquet(v3_path)
print(f"V3 schema: {v3_df.columns}")
print("(discount removed, loyalty_points added)")
print()

# Read all versions together
all_versions = spark.read.option("mergeSchema", True) \
                         .parquet(v1_path, v2_path, v3_path)

print(f"All versions schema: {all_versions.columns}")
print(f"Total rows: {all_versions.count():,}")
print()
all_versions.select("order_id","discount","country","loyalty_points").show(8)
print("V1 rows: null for discount, country, loyalty_points")
print("V2 rows: has discount/country, null for loyalty_points")
print("V3 rows: null for discount, has country and loyalty_points")

26/04/10 21:17:13 WARN TaskSetManager: Stage 19 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


V3 schema: ['order_id', 'customer_id', 'product', 'category', 'revenue', 'order_date', 'country', 'loyalty_points']
(discount removed, loyalty_points added)

All versions schema: ['order_id', 'customer_id', 'product', 'category', 'revenue', 'order_date', 'discount', 'country', 'loyalty_points']
Total rows: 600,000

+--------+--------+-------+--------------+
|order_id|discount|country|loyalty_points|
+--------+--------+-------+--------------+
|       1|    NULL|     UK|          1954|
|       2|    NULL|     AU|          1740|
|       3|    NULL|     JP|        145345|
|       4|    NULL|     US|         19487|
|       5|    NULL|     FR|         18620|
|       6|    NULL|     BR|         75011|
|       7|    NULL|     DE|         45325|
|       8|    NULL|     CA|         49624|
+--------+--------+-------+--------------+
only showing top 8 rows
V1 rows: null for discount, country, loyalty_points
V2 rows: has discount/country, null for loyalty_points
V3 rows: null for discount, has coun

## Step 3 — Type Promotion: Safe vs Unsafe

Some type changes are safe (widening), others corrupt data.

| Change | Safe? | Notes |
|---|---|---|
| `INT` → `LONG` | ✅ Safe | Values always fit |
| `FLOAT` → `DOUBLE` | ✅ Safe | More precision |
| `INT` → `DOUBLE` | ✅ Safe | Widening |
| `LONG` → `INT` | ❌ Unsafe | Overflow possible |
| `DOUBLE` → `FLOAT` | ❌ Unsafe | Precision loss |
| `STRING` → `INT` | ❌ Unsafe | Parse errors |
| `INT` → `STRING` | ⚠️ Works | But loses numeric semantics |


In [5]:
# Safe: INT → LONG
int_df = df.select("order_id","quantity").withColumn("quantity", F.col("quantity").cast("int"))
int_path = f"{BASE_PATH}/type_int"
int_df.write.mode("overwrite").parquet(int_path)

long_df = df.select("order_id","quantity").withColumn("quantity", F.col("quantity").cast("long"))
long_path = f"{BASE_PATH}/type_long"
long_df.write.mode("overwrite").parquet(long_path)

# Read INT file with LONG schema (safe widening)
long_schema = spark.read.parquet(long_path).schema
result_safe = spark.read.schema(long_schema).parquet(int_path)
print("INT → LONG (safe widening):")
result_safe.printSchema()
result_safe.show(3)
print("✅ All values preserved correctly")

# Unsafe: LONG → INT — Spark 4.x REJECTS this with PARQUET_COLUMN_DATA_TYPE_MISMATCH
print("\nLONG → INT (unsafe narrowing):")
try:
    int_schema = spark.read.parquet(int_path).schema
    result_unsafe = spark.read.schema(int_schema).parquet(long_path)
    result_unsafe.show(3)
    print("⚠️  Values may be silently truncated — avoid type narrowing")
except Exception as e:
    print(f"❌ Spark 4.x rejects LONG→INT narrowing: {type(e).__name__}")
    print("   PARQUET_COLUMN_DATA_TYPE_MISMATCH: INT64 cannot be read as int")
    print()
    print("   Rule: Parquet type narrowing is NEVER safe.")
    print("   INT→LONG:    ✅  (widening — all INT values fit in LONG)")
    print("   LONG→INT:    ❌  (narrowing — large values would overflow)")
    print("   FLOAT→DOUBLE:✅  (widening)")
    print("   DOUBLE→FLOAT:❌  (narrowing — precision loss)")

26/04/10 21:17:15 WARN TaskSetManager: Stage 25 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:17:15 WARN TaskSetManager: Stage 26 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


INT → LONG (safe widening):
root
 |-- order_id: long (nullable = true)
 |-- quantity: long (nullable = true)

+--------+--------+
|order_id|quantity|
+--------+--------+
|       1|       4|
|       2|       1|
|       3|       9|
+--------+--------+
only showing top 3 rows
✅ All values preserved correctly

LONG → INT (unsafe narrowing):


26/04/10 21:17:17 WARN TaskSetManager: Lost task 0.0 in stage 30.0 (TID 64) (172.18.0.4 executor 1): org.apache.spark.SparkException: [FAILED_READ_FILE.PARQUET_COLUMN_DATA_TYPE_MISMATCH] Encountered error while reading file file:///workspace/data/parquet_basics/schema_evolution/type_long/part-00000-da2887f1-ac5a-41f8-83e3-100b72412f68-c000.snappy.parquet. Data type mismatches when reading Parquet column [quantity]. Expected Spark type int, actual Parquet type INT64. SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.parquetColumnDataTypeMismatchError(QueryExecutionErrors.scala:847)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:138)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:142)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:695)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCo

❌ Spark 4.x rejects LONG→INT narrowing: Py4JJavaError
   PARQUET_COLUMN_DATA_TYPE_MISMATCH: INT64 cannot be read as int

   Rule: Parquet type narrowing is NEVER safe.
   INT→LONG:    ✅  (widening — all INT values fit in LONG)
   LONG→INT:    ❌  (narrowing — large values would overflow)
   FLOAT→DOUBLE:✅  (widening)
   DOUBLE→FLOAT:❌  (narrowing — precision loss)


26/04/10 21:17:18 ERROR TaskSetManager: Task 0 in stage 30.0 failed 4 times; aborting job


## Step 4 — Renaming Columns

Parquet doesn't store column aliases — renaming breaks backward compatibility.
The safe approach: add new column, migrate, then drop old.


In [6]:
# Problem: simply renaming breaks reading old files
old_df = df.select("order_id", "customer_id", "revenue")
new_df = df.select("order_id",
                   F.col("customer_id").alias("cust_id"),   # renamed!
                   "revenue")

old_path = f"{BASE_PATH}/rename_old"
new_path = f"{BASE_PATH}/rename_new"
old_df.write.mode("overwrite").parquet(old_path)
new_df.write.mode("overwrite").parquet(new_path)

# Read both — columns don't align!
print("Problem: old file has 'customer_id', new file has 'cust_id'")
combined_broken = spark.read.option("mergeSchema", True).parquet(old_path, new_path)
print(f"Combined schema: {combined_broken.columns}")
combined_broken.show(5)
print("Both columns present — old rows have null cust_id, new rows have null customer_id!")
print()

# Safe rename: keep old name as alias, migrate data
print("Safe migration pattern:")
print("  1. Add new column name (keep old too)")
print("  2. Backfill new column from old")
print("  3. Once all files updated: drop old column")
migration_df = old_df.withColumn("cust_id", F.col("customer_id"))
print(f"Migration schema: {migration_df.columns}")

26/04/10 21:17:18 WARN TaskSetManager: Stage 31 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:17:19 WARN TaskSetManager: Stage 32 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


Problem: old file has 'customer_id', new file has 'cust_id'
Combined schema: ['order_id', 'cust_id', 'revenue', 'customer_id']
+--------+-------+--------+-----------+
|order_id|cust_id| revenue|customer_id|
+--------+-------+--------+-----------+
|       1|   NULL|  195.44|       9145|
|       2|   NULL|  174.07|       1740|
|       3|   NULL|14534.56|      45754|
|       4|   NULL| 1948.71|      22542|
|       5|   NULL| 1862.01|      40536|
+--------+-------+--------+-----------+
only showing top 5 rows
Both columns present — old rows have null cust_id, new rows have null customer_id!

Safe migration pattern:
  1. Add new column name (keep old too)
  2. Backfill new column from old
  3. Once all files updated: drop old column
Migration schema: ['order_id', 'customer_id', 'revenue', 'cust_id']


## Summary: Schema Evolution Rules

| Change | Safe | How |
|---|---|---|
| Add column | ✅ Yes | Old files return null for new column |
| Drop column | ✅ Yes | New files don't write it; old files still have it |
| Widen type (int→long) | ✅ Yes | Read old files with wider schema |
| Narrow type (long→int) | ❌ No | Risk of overflow/truncation |
| Rename column | ⚠️ Care | Add+migrate+drop pattern |

### mergeSchema quick reference
```python
# Read multiple versions with different schemas
spark.read.option("mergeSchema", True).parquet(v1_path, v2_path, v3_path)

# Or from one directory containing mixed-version files
spark.read.option("mergeSchema", True).parquet("/data/table/")
```

**Rule:** Use Iceberg or Delta for production tables with frequent schema changes
— they have first-class schema evolution support with full history.
